# Waterway and Railway Data Preparation

This notebook extracts and cleans OpenStreetMap waterway and railway networks and groups relevant features into broader categories for use as spatial context in building classification.

**Input**
- Raw OpenStreetMap data for Germany.

**Output**
- Cleaned and categorized waterway and railway network data for subsequent feature engineering.

In [ ]:
import pyrosm
import os

ROOT_DIR = '/fast/home/o-olajuyigbe/osm_project'
DATA_DIR = os.path.join(ROOT_DIR, 'data')
PBF_FILE = os.path.join(DATA_DIR, 'raw', 'germany-latest.osm.pbf')

osm = pyrosm.OSM(PBF_FILE)

In [6]:
# ── Waterways ──────────────────────────────────────────────────────────
print("Extracting waterways...")

waterways_raw = osm.get_data_by_custom_criteria(
    custom_filter={'waterway': ['river', 'canal', 'stream', 'drain', 'ditch']}
)

print(f"Raw waterway rows  : {len(waterways_raw):,}")
print(waterways_raw.geometry.geom_type.value_counts())
print(waterways_raw.columns.tolist())

Extracting waterways...
Raw waterway rows  : 1,558,221
LineString         781998
MultiLineString    775872
Polygon               192
Point                 159
Name: count, dtype: int64
['id', 'tags', 'changeset', 'lon', 'visible', 'lat', 'timestamp', 'version', 'waterway', 'geometry', 'osm_type', 'canal', 'ditch', 'fish_pass', 'stream', 'waterfall', 'weir']


In [7]:
waterways_raw.to_parquet(os.path.join(DATA_DIR, 'raw', 'germany_waterways.parquet'), index=False)

In [8]:
# Keep only line geometries (same as roads)
VALID_WATERWAY_GEOMS = {'LineString', 'MultiLineString'}
waterways = waterways_raw[
    waterways_raw.geometry.geom_type.isin(VALID_WATERWAY_GEOMS)
].copy()

print(f"After geometry filter: {len(waterways):,}")

# ── Category mapping ───────────────────────────────────────────────────
# Two categories: major (river/canal) vs minor (stream/drain/ditch)
# Major waterways are the ones that matter for industrial location
WATERWAY_MAP = {
    'river'  : 'major_waterway',
    'canal'  : 'major_waterway',
    'stream' : 'minor_waterway',
    'drain'  : 'minor_waterway',
    'ditch'  : 'minor_waterway',
}

waterways['waterway_category'] = waterways['waterway'].map(WATERWAY_MAP)

# Drop unmapped
waterways = waterways[waterways['waterway_category'].notna()].copy()

print(waterways['waterway_category'].value_counts())
print(f"Final waterway rows: {len(waterways):,}")

After geometry filter: 1,557,870
waterway_category
minor_waterway    1534402
major_waterway      23468
Name: count, dtype: int64
Final waterway rows: 1,557,870


In [9]:
waterways.head()

,id,tags,changeset,lon,visible,lat,timestamp,version,waterway,geometry,osm_type,canal,ditch,fish_pass,stream,waterfall,weir,waterway_category
159,2976788,"{""visible"":false,""name"":""Moorburger Landscheid...",NaN,NaN,NaN,NaN,1739630073,27,canal,"MULTILINESTRING ((9.93705 53.4759, 9.93699 53....",way,None,None,None,None,None,None,major_waterway
160,3563596,"{""visible"":false,""name"":""Becken V"",""ship"":""yes...",NaN,NaN,NaN,NaN,1650292725,4,canal,"LINESTRING (8.31363 49.01604, 8.32605 49.01049)",way,None,None,None,None,None,None,major_waterway
161,3563597,"{""visible"":false,""name"":""Becken IV"",""ship"":""ye...",NaN,NaN,NaN,NaN,1650292725,4,canal,"LINESTRING (8.3219 49.01582, 8.33358 49.01049)",way,None,None,None,None,None,None,major_waterway
162,3563598,"{""visible"":false,""name"":""Becken III"",""ship"":""y...",NaN,NaN,NaN,NaN,1650292725,4,canal,"MULTILINESTRING ((8.32874 49.01564, 8.33138 49...",way,None,None,None,None,None,None,major_waterway
163,3563599,"{""visible"":false,""name"":""Becken II"",""ship"":""ye...",NaN,NaN,NaN,NaN,1650292725,5,canal,"MULTILINESTRING ((8.32874 49.01564, 8.33075 49...",way,None,None,None,None,None,None,major_waterway


In [10]:
# Keep only the columns needed for feature engineering
waterways_slim = waterways[['id', 'geometry', 'waterway_category']].copy()

# Fix invalid geometries
invalid = ~waterways_slim.geometry.is_valid
if invalid.any():
    waterways_slim.loc[invalid, 'geometry'] = waterways_slim.loc[invalid, 'geometry'].buffer(0)
    waterways_slim = waterways_slim[waterways_slim.geometry.is_valid].copy()
    print(f"Fixed {invalid.sum()} invalid geometries")

waterways_slim.to_parquet(
    os.path.join(DATA_DIR, 'processed', 'germany_waterways_mapped.parquet'),
    index=False
)
print(f"Saved germany_waterways_mapped.parquet — {len(waterways_slim):,} rows")
print(f"CRS: {waterways_slim.crs}")

Fixed 101 invalid geometries
Saved germany_waterways_mapped.parquet — 1,557,870 rows
CRS: epsg:4326


In [11]:
# ── Railways ───────────────────────────────────────────────────────────
print("Extracting railways...")

railways_raw = osm.get_data_by_custom_criteria(
    custom_filter={'railway': ['rail', 'light_rail', 'narrow_gauge', 'tram', 'subway']}
)

print(f"Raw railway rows   : {len(railways_raw):,}")
print(railways_raw.geometry.geom_type.value_counts())

Extracting railways...
Raw railway rows   : 271,016
MultiLineString    197135
LineString          73812
Polygon                69
Name: count, dtype: int64


In [12]:
railways_raw.to_parquet(os.path.join(DATA_DIR, 'raw', 'germany_railways.parquet'), index=False)

In [13]:
VALID_RAILWAY_GEOMS = {'LineString', 'MultiLineString'}
railways = railways_raw[
    railways_raw.geometry.geom_type.isin(VALID_RAILWAY_GEOMS)
].copy()

print(f"After geometry filter: {len(railways):,}")

# ── Category mapping ───────────────────────────────────────────────────
# Two categories: heavy rail (freight lines — industrial signal) vs light rail
RAILWAY_MAP = {
    'rail'          : 'heavy_rail',    # main line — freight + passenger, strong industrial signal
    'narrow_gauge'  : 'heavy_rail',    # industrial narrow gauge
    'light_rail'    : 'light_rail',    # urban, weaker industrial signal
    'tram'          : 'light_rail',
    'subway'        : 'light_rail',
}

railways['railway_category'] = railways['railway'].map(RAILWAY_MAP)
railways = railways[railways['railway_category'].notna()].copy()

print(railways['railway_category'].value_counts())
print(f"Final railway rows: {len(railways):,}")

After geometry filter: 270,947
railway_category
heavy_rail    230412
light_rail     40535
Name: count, dtype: int64
Final railway rows: 270,947


In [14]:
railways_slim = railways[['id', 'geometry', 'railway_category']].copy()

invalid = ~railways_slim.geometry.is_valid
if invalid.any():
    railways_slim.loc[invalid, 'geometry'] = railways_slim.loc[invalid, 'geometry'].buffer(0)
    railways_slim = railways_slim[railways_slim.geometry.is_valid].copy()

railways_slim.to_parquet(
    os.path.join(DATA_DIR, 'processed', 'germany_railways_mapped.parquet'),
    index=False
)
print(f"Saved germany_railways_mapped.parquet — {len(railways_slim):,} rows")
print(f"CRS: {railways_slim.crs}")

Saved germany_railways_mapped.parquet — 270,947 rows
CRS: epsg:4326
